# License Plate Detector Training: YOLOX-Nano (Apache 2.0)

[UPDATE FROM: 2/11 - Training YOLOX (Under apache 2.0 license)]

This notebook trains a single-class license plate detector using **Megvii YOLOX-Nano** under the **Apache 2.0 License**.
It incorporates the exact fleet balancing (1,400 cars, 1,400 motorbikes), Albumentations skew augmentations, and negative sample injection from the prior production pipeline.

## 1. Verify GPU Hardware Acceleration

In [1]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Active GPU:", torch.cuda.get_device_name(0))

Mon Nov  2 05:24:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies & Clone Megvii YOLOX (Apache 2.0)

In [2]:
# 1. Install supporting libraries for dataset merging, augmentation, and ONNX export
!pip install -q roboflow albumentations pycocotools onnx onnxruntime onnxsim pyyaml loguru tabulate thop

# 2. Clone official Megvii BaseDetection YOLOX repository (Apache 2.0)
import os
if not os.path.exists("YOLOX") and not os.path.exists("yolox"):
    !git clone https://github.com/Megvii-BaseDetection/YOLOX.git

if os.path.exists("YOLOX"):
    %cd YOLOX

# 3. Strip obsolete 2021 ONNX pins from YOLOX requirements (fixes onnx-simplifier build failure)
!sed -i "/onnx/d" requirements.txt

# 4. Install YOLOX package cleanly with modern pre-installed packages
!pip install -v -e . --no-build-isolation --no-deps


Cloning into 'YOLOX'...
remote: Enumerating objects: 1940, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 1940 (delta 172), reused 164 (delta 164), pack-reused 1737 (from 1)
Receiving objects: 100% (1940/1940), 7.55 MiB | 35.13 MiB/s, done.
Resolving deltas: 100% (1171/1171), done.
/content/YOLOX
Using pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
Obtaining file:///content/YOLOX
  Running command Checking if build backend supports build_editable
  Checking if build backend supports build_editable ... done
  Running command Preparing editable metadata (pyproject.toml)
  /usr/local/lib/python3.13/dist-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
  !!

          ********************************************************************************
          Please consider removing the following classifiers in favor of a SPDX license expression:

## 3. Download Official Pretrained Base Weights (Apache 2.0)

In [3]:
import urllib.request
import os

weights_url = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth"
weights_file = "yolox_nano.pth"

if not os.path.exists(weights_file):
    print(f"Downloading {weights_file}...")
    urllib.request.urlretrieve(weights_url, weights_file)
    print("[OK] Base weights downloaded.")
else:
    print("[OK] Base weights already present.")

[OK] Base weights downloaded.


## 4. Download Datasets from Roboflow

In [4]:
import os, shutil, random, glob, yaml
from roboflow import Roboflow

rf = Roboflow(api_key="")  # Replace with your actual Roboflow API key

# Car fleet dataset
car_dominant_project = rf.workspace("alphajack2006-gmail-com").project("vietnamese-car-license-plate-vlbzz")
car_dominant_version = car_dominant_project.version(1)
car_dominant_dataset = car_dominant_version.download("yolov8")

# Motorcycle fleet dataset
moto_dominant_project = rf.workspace("alphajack2006-gmail-com").project("motorcycle-license-plate-sksvk")
moto_dominant_version = moto_dominant_project.version(1)
moto_dominant_dataset = moto_dominant_version.download("yolov8")

# Negative background / pavement dataset
neg_sam_project = rf.workspace("alphajack2006-gmail-com").project("road-and-pavement-hc3qg-qsmcl")
neg_sam_version = neg_sam_project.version(1)
neg_sam_dataset = neg_sam_version.download("yolov8")

car_dir = car_dominant_dataset.location
moto_dir = moto_dominant_dataset.location
unified_dir = "/content/dataset_unified"

print(f"Car dataset downloaded to: {car_dir}")
print(f"Moto dataset downloaded to: {moto_dir}")
print(f"Unified dataset will be stored in: {unified_dir}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Vietnamese-Car-License-Plate--1 in yolov8:: 100%|██████████| 16513/16513 [00:03<00:00, 4456.94it/s]


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Motorcycle-license-plate-1 in yolov8:: 100%|██████████| 3500/3500 [00:00<00:00, 11487.47it/s]

loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to road-and-pavement-1 in yolov8:: 100%|██████████| 5584/5584 [00:04<00:00, 1246.23it/s]


Car dataset downloaded to: /content/YOLOX/Vietnamese-Car-License-Plate--1
Moto dataset downloaded to: /content/YOLOX/Motorcycle-license-plate-1
Unified dataset will be stored in: /content/dataset_unified


## 5. Merge Datasets (Balanced 1:1 Car & Motorcycle Fleet)

In [5]:
splits = ["train", "valid"]
for split in splits:
    os.makedirs(os.path.join(unified_dir, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(unified_dir, split, "labels"), exist_ok=True)

def merge_datasets(src_dir, prefix, max_train_samples=None, seed=40):
    for split in splits:
        img_src = os.path.join(src_dir, split, "images")
        lbl_src = os.path.join(src_dir, split, "labels")

        if not os.path.exists(img_src):
            alt_split = "valid"
            img_src = os.path.join(src_dir, alt_split, "images")
            lbl_src = os.path.join(src_dir, alt_split, "labels")

        valid_exts = (".jpg", ".jpeg", ".png")
        all_images = [f for f in os.listdir(img_src) if f.lower().endswith(valid_exts)]

        if split == "train" and max_train_samples and len(all_images) > max_train_samples:
            random.seed(seed)
            selected_images = random.sample(all_images, max_train_samples)
            print(f"[{prefix.upper()} - {split} downsampled from {len(all_images)} to {len(selected_images)} images]")
        else:
            selected_images = all_images
            print(f"[{prefix.upper()} - {split} importing all {len(selected_images)} images]")

        for filename in selected_images:
            base_name, ext = os.path.splitext(filename)
            lbl_file = f"{base_name}.txt"

            src_img_path = os.path.join(img_src, filename)
            src_lbl_path = os.path.join(lbl_src, lbl_file)

            dest_img_path = os.path.join(unified_dir, split, "images", f"{prefix}_{filename}")
            dest_lbl_path = os.path.join(unified_dir, split, "labels", f"{prefix}_{lbl_file}")

            shutil.copy(src_img_path, dest_img_path)

            if os.path.exists(src_lbl_path):
                with open(src_lbl_path, "r", encoding="utf-8") as file:
                    lines = file.readlines()

                remapped = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        parts[0] = "0"  # Single class: 0 (plate)
                        remapped.append(" ".join(parts) + " ")

                with open(dest_lbl_path, "w", encoding="utf-8") as file:
                    file.writelines(remapped)
            else:
                open(dest_lbl_path, "w", encoding="utf-8").close()

# Execute 1:1 balanced merge
merge_datasets(car_dir, "car", max_train_samples=1400, seed=42)
merge_datasets(moto_dir, "moto", max_train_samples=1400, seed=42)
print("[OK] Balanced dataset merge completed.")

[CAR - train downsampled from 5778 to 1400 images]
[CAR - valid importing all 2477 images]
[MOTO - train importing all 1397 images]
[MOTO - valid importing all 174 images]
[OK] Balanced dataset merge completed.


## 6. Albumentations Skew & Perspective Augmentation

In [6]:
import albumentations as A
import cv2

def generate_skewed_samples(unified_dir, prefix, sample_count=140):
    img_dir = os.path.join(unified_dir, "train", "images")
    lbl_dir = os.path.join(unified_dir, "train", "labels")

    transform = A.Compose([
        A.Perspective(scale=(0.02, 0.04), p=0.4, keep_size=True),
        A.ShiftScaleRotate(shift_limit=0.03, scale_limit=0.05, rotate_limit=10, p=0.7, border_mode=cv2.BORDER_REFLECT_101),
        A.Affine(shear=(-4, 4), p=0.4)
    ], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"], min_visibility=0.3))

    candidates = [f for f in os.listdir(img_dir) if f.startswith(prefix) and f.lower().endswith((".jpg", ".jpeg", ".png"))]
    selected = random.sample(candidates, min(sample_count, len(candidates)))

    generated = 0
    for fname in selected:
        stem, ext = os.path.splitext(fname)
        lbl_file = f"{stem}.txt"
        img_path = os.path.join(img_dir, fname)
        lbl_path = os.path.join(lbl_dir, lbl_file)

        if not os.path.exists(lbl_path):
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue

        bboxes = []
        class_labels = []
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    class_labels.append(int(parts[0]))
                    bboxes.append([float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])])

        if not bboxes:
            continue

        transformed = transform(image=img, bboxes=bboxes, class_labels=class_labels)
        if transformed["bboxes"]:
            out_img_name = f"skew_{fname}"
            out_lbl_name = f"skew_{stem}.txt"
            cv2.imwrite(os.path.join(img_dir, out_img_name), transformed["image"])
            with open(os.path.join(lbl_dir, out_lbl_name), "w") as f:
                for cls, box in zip(transformed["class_labels"], transformed["bboxes"]):
                    f.write(f"{int(float(cls))} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} ")
            generated += 1

    print(f"[{prefix.upper()}] Generated {generated} skewed training samples.")

# Generate 140 skewed cars and 140 skewed motorbikes
generate_skewed_samples(unified_dir, prefix="car", sample_count=140)
generate_skewed_samples(unified_dir, prefix="moto", sample_count=140)

ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.


[CAR] Generated 140 skewed training samples.
[MOTO] Generated 140 skewed training samples.


## 7. Inject External Negative Samples (Pavement / Roads)

In [7]:
def add_external_as_negatives(dataset_dir, unified_dir, max_samples=120, seed=42):
    dest_img = os.path.join(unified_dir, "train", "images")
    dest_lbl = os.path.join(unified_dir, "train", "labels")

    src_img_dir = os.path.join(dataset_dir, "train", "images")
    if not os.path.exists(src_img_dir):
        src_img_dir = os.path.join(dataset_dir, "images")
        if not os.path.exists(src_img_dir):
            src_img_dir = dataset_dir

    valid_exts = (".jpg", ".jpeg", ".png", ".bmp")
    all_images = [f for f in os.listdir(src_img_dir) if f.lower().endswith(valid_exts)]

    random.seed(seed)
    count = min(len(all_images), max_samples)
    selected = random.sample(all_images, count)

    for filename in selected:
        base_name, _ = os.path.splitext(filename)
        shutil.copy(os.path.join(src_img_dir, filename), os.path.join(dest_img, f"neg_{filename}"))
        # Create 0-byte label file (pure negative sample)
        open(os.path.join(dest_lbl, f"neg_{base_name}.txt"), "w").close()

    print(f"[OK] Injected {count} negative background samples into training set.")

add_external_as_negatives(neg_sam_dataset.location, unified_dir, max_samples=120)

[OK] Injected 120 negative background samples into training set.


## 8. Convert Unified YOLO Dataset to COCO JSON Format for YOLOX

YOLOX native data loader expects annotations in COCO JSON format ( and ). This cell converts the YOLO coordinates to COCO format automatically.

In [8]:
import json
from PIL import Image

coco_root = "/content/dataset_coco"
os.makedirs(os.path.join(coco_root, "annotations"), exist_ok=True)

def convert_yolo_to_coco(unified_dir, split_name, coco_split_name):
    src_img_dir = os.path.join(unified_dir, split_name, "images")
    src_lbl_dir = os.path.join(unified_dir, split_name, "labels")
    dest_img_dir = os.path.join(coco_root, coco_split_name)
    os.makedirs(dest_img_dir, exist_ok=True)

    coco_data = {
        "images": [],
        "annotations": [],
        "categories": [{"id": 1, "name": "plate", "supercategory": "none"}]
    }

    valid_exts = (".jpg", ".jpeg", ".png")
    img_files = [f for f in os.listdir(src_img_dir) if f.lower().endswith(valid_exts)]
    img_files.sort()

    ann_id = 1
    for img_id, filename in enumerate(img_files, 1):
        src_path = os.path.join(src_img_dir, filename)
        dest_path = os.path.join(dest_img_dir, filename)
        shutil.copy(src_path, dest_path)

        with Image.open(src_path) as im:
            width, height = im.size

        coco_data["images"].append({
            "id": img_id,
            "file_name": filename,
            "width": width,
            "height": height
        })

        stem, _ = os.path.splitext(filename)
        lbl_path = os.path.join(src_lbl_dir, f"{stem}.txt")
        if os.path.exists(lbl_path):
            with open(lbl_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                        abs_x = (cx - bw / 2.0) * width
                        abs_y = (cy - bh / 2.0) * height
                        abs_w = bw * width
                        abs_h = bh * height
                        coco_data["annotations"].append({
                            "id": ann_id,
                            "image_id": img_id,
                            "category_id": 1,
                            "bbox": [round(abs_x, 2), round(abs_y, 2), round(abs_w, 2), round(abs_h, 2)],
                            "area": round(abs_w * abs_h, 2),
                            "iscrowd": 0
                        })
                        ann_id += 1

    ann_file = os.path.join(coco_root, "annotations", f"instances_{coco_split_name}.json")
    with open(ann_file, "w") as f:
        json.dump(coco_data, f)
    print(f"[OK] Converted {len(img_files)} images and {ann_id - 1} boxes to {ann_file}")

convert_yolo_to_coco(unified_dir, "train", "train2017")
convert_yolo_to_coco(unified_dir, "valid", "val2017")

[OK] Converted 3197 images and 3060 boxes to /content/dataset_coco/annotations/instances_train2017.json
[OK] Converted 2651 images and 2616 boxes to /content/dataset_coco/annotations/instances_val2017.json


## 9. Define YOLOX Experiment Configuration (exps/custom/yolox_nano_plate.py)

In YOLOX, an Exp (Experiment) file is a Python configuration file (equivalent to data.yaml in Ultralytics) that specifies dataset paths, epoch counts, and single-class setup.

In [9]:
exp_config = """import os
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super(Exp, self).__init__()
        # Architecture parameters for YOLOX-Nano
        self.depth = 0.33
        self.width = 0.25
        self.scale = (0.5, 1.5)
        self.exp_name = os.path.split(os.path.realpath(__file__))[1].split(".")[0]

        # Single class: license plate
        self.num_classes = 1

        # Training hyperparameters (40 epochs baseline)
        self.max_epoch = 40
        self.data_num_workers = 4
        self.eval_interval = 5
        self.input_size = (640, 640)
        self.test_size = (640, 640)
        self.basic_lr_per_img = 0.01 / 64.0

        # COCO dataset paths
        self.data_dir = "/content/dataset_coco"
        self.train_ann = "instances_train2017.json"
        self.val_ann = "instances_val2017.json"
"""

os.makedirs("exps/custom", exist_ok=True)
with open("exps/custom/yolox_nano_plate.py", "w", encoding="utf-8") as f:
    f.write(exp_config)
print("[OK] Custom experiment configuration saved to exps/custom/yolox_nano_plate.py")

[OK] Custom experiment configuration saved to exps/custom/yolox_nano_plate.py


## 10. Fine-Tune YOLOX-Nano (40 Epochs with FP16 Mixed Precision)

In [10]:
# Launch fine-tuning with pretrained base checkpoint (takes ~15-20 min on Colab T4 GPU)
# TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 enables loading legacy YOLOX weights in PyTorch 2.6+
%env TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python -m yolox.tools.train -f exps/custom/yolox_nano_plate.py -d 1 -b 16 --fp16 -c yolox_nano.pth

env: TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1
I0000 00:00:1793597176.213644    1380 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793597178.642111    1380 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/content/YOLOX/yolox/core/trainer.py:47: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=args.fp16)
2026-11-02 05:26:20 | INFO     | yolox.core.trainer:132 - args: Namespace(experiment_name='yolox_nano_plate', name=None, dist_backend='nccl', dist_url=None, batch_size=16, devices=1, exp_file='exps/custom/yolox_nano_plate.py', resume=False, ckpt='yolox_nano.pth', start_epoch=None, num_machines=1, machin

## 11. Evaluate Trained Checkpoint (mAP50 Validation)

In [11]:
best_checkpoint = "YOLOX_outputs/yolox_nano_plate/best_ckpt.pth"
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python -m yolox.tools.eval -f exps/custom/yolox_nano_plate.py -c YOLOX_outputs/yolox_nano_plate/best_ckpt.pth -b 16 -d 1 --conf 0.25

I0000 00:00:1793601732.039817   21772 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793601734.951632   21772 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-02 06:42:16 | INFO     | __main__:139 - Args: Namespace(experiment_name='yolox_nano_plate', name=None, dist_backend='nccl', dist_url=None, batch_size=16, devices=1, num_machines=1, machine_rank=0, exp_file='exps/custom/yolox_nano_plate.py', ckpt='YOLOX_outputs/yolox_nano_plate/best_ckpt.pth', conf=0.25, nms=None, tsize=None, seed=None, fp16=False, fuse=False, trt=False, legacy=False, test=False, speed=False, opts=[])
2026-11-02 06:42:17 | INFO     | __main__:149 - Model Summary: Params: 2.24M, Gflops: 6.93
2026-11-02 06:42:17 | INFO     | __main__:150 - Model

## 12. Export to Clean-Room detector.onnx

The --decode_in_inference flag embeds anchor decoding directly into the ONNX graph so src/eiu_lpr/detector.py can ingest the output with zero extra compute.

In [18]:
import os
import torch
from yolox.exp import get_exp

# 1. Load experiment config and instantiate architecture
exp = get_exp("exps/custom/yolox_nano_plate.py")
model = exp.get_model()

# 2. Load the trained 18MB checkpoint
ckpt = torch.load(
    "YOLOX_outputs/yolox_nano_plate/best_ckpt.pth",
    map_location="cpu",
    weights_only=False,
)
model.load_state_dict(ckpt["model"])
model.eval()

# 3. Embed bounding box grid decoding into the ONNX graph
model.head.decode_in_inference = True

# 4. Export using TorchScript tracer (dynamo=False bypasses onnxscript dependency)
dummy_input = torch.randn(1, 3, 640, 640)
torch.onnx.export(
    model,
    dummy_input,
    "detector.onnx",
    input_names=["images"],
    output_names=["output"],
    opset_version=11,
    dynamo=False,
)

size_mb = os.path.getsize("detector.onnx") / (1024 * 1024)
print(f"[OK] detector.onnx exported successfully: {size_mb:.2f} MB")

You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html


[OK] detector.onnx exported successfully: 8.67 MB


In [19]:
!ls -lh YOLOX_outputs/yolox_nano_plate/

total 418M
-rw-r--r-- 1 root root  18M Nov  2 06:32 best_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 05:49 epoch_10_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:00 epoch_15_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:11 epoch_20_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:21 epoch_25_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:22 epoch_26_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:24 epoch_27_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:25 epoch_28_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:26 epoch_29_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:28 epoch_30_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:29 epoch_31_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:31 epoch_32_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:32 epoch_33_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:33 epoch_34_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:35 epoch_35_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:36 epoch_36_ckpt.pth
-rw-r--r-- 1 root root  18M Nov  2 06:37 epoch_37

In [20]:
import os

print("Working directory:", os.getcwd())
print(
    "best_ckpt.pth exists?:",
    os.path.exists("YOLOX_outputs/yolox_nano_plate/best_ckpt.pth"),
)

# List all files inside YOLOX_outputs
if os.path.exists("YOLOX_outputs"):
  for root, dirs, files in os.walk("YOLOX_outputs"):
    print(root, files)
else:
  print("YOLOX_outputs directory does NOT exist!")

Working directory: /content/YOLOX
best_ckpt.pth exists?: True
YOLOX_outputs []
YOLOX_outputs/yolox_nano_plate ['epoch_35_ckpt.pth', 'epoch_30_ckpt.pth', 'last_mosaic_epoch_ckpt.pth', 'epoch_31_ckpt.pth', 'epoch_26_ckpt.pth', 'epoch_40_ckpt.pth', 'epoch_33_ckpt.pth', 'epoch_28_ckpt.pth', 'epoch_37_ckpt.pth', 'epoch_27_ckpt.pth', 'epoch_39_ckpt.pth', 'epoch_29_ckpt.pth', 'epoch_10_ckpt.pth', 'best_ckpt.pth', 'epoch_36_ckpt.pth', 'last_epoch_ckpt.pth', 'epoch_20_ckpt.pth', 'val_log.txt', 'epoch_15_ckpt.pth', 'latest_ckpt.pth', 'epoch_25_ckpt.pth', 'epoch_32_ckpt.pth', 'epoch_5_ckpt.pth', 'epoch_38_ckpt.pth', 'epoch_34_ckpt.pth', 'train_log.txt']
YOLOX_outputs/yolox_nano_plate/tensorboard ['events.out.tfevents.1793597185.9ed62ee1a325.1380.0']


## 13. Verify ONNX Output Tensor & Shapes

In [21]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession("detector.onnx", providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
print("Input Name:", input_name, "Shape:", session.get_inputs()[0].shape)
print("Output Name:", output_name, "Shape:", session.get_outputs()[0].shape)

dummy_img = np.zeros((1, 3, 640, 640), dtype=np.float32)
preds = session.run([output_name], {input_name: dummy_img})[0]
print("Output Tensor Shape:", preds.shape)
print("Detected Attributes:", preds.shape[-1], "(cx, cy, bw, bh, obj_conf, cls_conf)")

Input Name: images Shape: [1, 3, 640, 640]
Output Name: output Shape: [1, 8400, 6]
Output Tensor Shape: (1, 8400, 6)
Detected Attributes: 6 (cx, cy, bw, bh, obj_conf, cls_conf)


## 14. Download Exported Model to Local Workspace

In [22]:
from google.colab import files
files.download("detector.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>